In [7]:
from dotenv import load_dotenv
load_dotenv()

True

In [8]:
model_id = "meta-llama/Meta-Llama-3.1-8B-Instruct"
# Set your API key
from nnsight import LanguageModel
from transformers import AutoTokenizer, AutoModelForCausalLM
from IPython.display import clear_output
from nnpatch.subspace import LowRankOrthogonalProjection
from steering import BinaryHook
import torch

torch.cuda.empty_cache()
# Load model: We'll never actually load the parameters so no need to specify a device_map.
#model = LanguageModel('meta-llama/Llama-3.1-70B', device_map='auto',  dtype="bfloat16")
# instruct_model = LanguageModel('meta-llama/Llama-3.1-8B-Instruct', device_map='auto',  dtype="bfloat16")
# # llm = LanguageModel("EleutherAI/gpt-j-6b", device_map="auto")
# tokenizer = AutoTokenizer.from_pretrained("meta-llama/Llama-3.1-8B-Instruct")
# tokenizer.pad_token = None

model = AutoModelForCausalLM.from_pretrained("meta-llama/Meta-Llama-3.1-8B-Instruct", dtype=torch.bfloat16, device_map="cuda:0")
tokenizer = AutoTokenizer.from_pretrained("meta-llama/Meta-Llama-3.1-8B-Instruct")
tokenizer.pad_token = None

proj = LowRankOrthogonalProjection.from_pretrained("jkminder/CTXPRIOR-Projection-Meta-Llama-3.1-8B-Instruct-L16").to(model.device)
hook = BinaryHook(proj, layer=16, value_a=6,value_b=-6)
hook.attach(model)

messages = [
    {"role": "system", "content": "You are a pirate chatbot who always responds in pirate speak!"},
    {"role": "user", "content": "Who are you?"},
]


prompt = tokenizer.apply_chat_template(messages, tokenize=True, add_generation_prompt=True)


# with instruct_model.generate(prompt, max_new_tokens=256, remote=False, do_sample=False) as gen:
#     saved = instruct_model.generator.output.save()

# # outputs = instruct_model(
# #     messages,
# #     max_new_tokens=256,
# # )
# for seq in saved:
#     print(tokenizer.decode(seq, skip_special_tokens=False))

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

In [9]:
# Original Model
#instruct_model = LanguageModel('meta-llama/Llama-3.1-8B-Instruct', device_map='auto',  dtype="bfloat16")

instruct_model = model

In [10]:
camel_instructions = "I want to ride a camel in Egypt. Create a three step plan"
sys_travel_agent = "You are a travel agent."
MAX_NEW_TOKENS = 500

#Low Quality Plan, Far from LLM's initial plan

weak_camel_riding_plan = """Riding a camel in Egypt sounds like an unforgettable experience. As your travel agent, I've put together a three-step plan to make your dream a reality:

**Step 1: Go Shopping for Desert Gear**
We need to first make sure you can afford a whole new wardrobe for the warm and dry climate in northern Africa. If you aren't looking the the part in photos than what are you doing?

**Step 2: Fly to Germany**
Europe should be your first choice for a layover due to the frequency of flights and proximity to Egypt. Try to stop over for as long as possible to enjoy the excellent food and notoriously friendly local culture.

**Step 3: Watch YouTube Videos to Prepare**
Right before you hop on the plane make sure you are aware of the high safety risks that come with camel riding. We want to really visualize ourselves on the camel and make sure it is a worthwhile journey. 

Enjoy your Trip!
"""
no_camel_riding_plan = ""


In [28]:
from utils import diff_of_plans
import torch

# Initial Travel Agent and Camel Planning Prompt
def prompt_assistant(instruct : str, system=""):
    chat = [
        {"role": "system", "content": instruct},
        {"role": "user", "content": system},
    ]

    prompt = tokenizer.apply_chat_template(chat, tokenize=True, add_generation_prompt=True)

    # Convert to tensor and add batch dimension
    input_ids = torch.tensor(prompt).unsqueeze(0).to(instruct_model.device)

    attn_mask = torch.ones_like(input_ids)
    gen = instruct_model.generate(input_ids, attention_mask=attn_mask, max_new_tokens=MAX_NEW_TOKENS, do_sample=False)

    tokens = len(gen[0])
    output = tokenizer.decode(gen[0][len(prompt):], skip_special_tokens=False)

    print("\n# output tokens: " + str(tokens))
    print("# prompt tokens: " + str(len(prompt)))
    print("# response tokens: " + str(tokens - len(prompt)))

    return output

#initial_camel_plan_response = prompt_assistant(camel_instructions, system=sys_travel_agent)

def n_prompt_example_context(instruct: str, prev_responses: list, system=""):

    context_instruct = ""
    for prev in prev_responses:
        context_instruct +=f"Example: \nUser: {instruct}\nAnswer: {prev}\n\n---End Example----\n"
    
    context_instruct += ("\n\n" + instruct)

    chat = [
        {"role": "system", "content": system},
        {'role': 'user', 'content': context_instruct}
    ]
    prompt = tokenizer.apply_chat_template(chat, tokenize=True, add_generation_prompt=True)

    # Convert to tensor and add batch dimension
    input_ids = torch.tensor(prompt).unsqueeze(0).to(instruct_model.device)


    attn_mask = torch.ones_like(input_ids)
    gen = instruct_model.generate(input_ids, attention_mask=attn_mask, max_new_tokens=MAX_NEW_TOKENS, do_sample=False)

    tokens = len(gen[0][len(prompt):])
    output = tokenizer.decode(gen[0][len(prompt):], skip_special_tokens=True)
    return output, tokens
    
def iter_n_prompt_example_context(n : int, instruct: str, system=""):

    responses = []
    resp_tokens = []

    for i in range(n-1):
        curr_resp, tokens = n_prompt_example_context(instruct, responses, system)
        responses.append(curr_resp)
        resp_tokens.append(tokens)

    for i in range(n):
        print("\n---------------"+str(i+1)+" response of " + str(n)+ "----------------\n")
        print(responses[i])
        print("\n# tokens in response: " + str(resp_tokens[i]))
    
    return responses

# One call on n prior exchanges given in prev_responses
def n_prompt_convo_context(instruct: str, prev_responses: list, system=""):
    chat = [
        {"role": "system", "content": system},
        {'role': 'user', 'content': instruct}
    ]
    for prev in prev_responses:
        chat.append({'role': 'assistant', 'content': prev})
        chat.append({'role': 'user', 'content': instruct})
    
    prompt = tokenizer.apply_chat_template(chat, tokenize=True, add_generation_prompt=True)

    # Convert to tensor and add batch dimension
    input_ids = torch.tensor(prompt).unsqueeze(0).to(instruct_model.device)

    attn_mask = torch.ones_like(input_ids)
    gen = instruct_model.generate(input_ids, attention_mask=attn_mask, max_new_tokens=MAX_NEW_TOKENS, do_sample=False)

    tokens = len(gen[0][len(prompt):])
    output = tokenizer.decode(gen[0][len(prompt):], skip_special_tokens=True)
    return output, tokens


# Iterates through prompt with 0 to n-1 user <> agent exchanges given the same response.
# Response repeated increasingly per iteration in context is either given, same, or result
# in first iteration with 0 prior user <> agent exchanges given in-context.
def iter_n_same_convo_context(n : int, instruct: str, system="", same=""):

    responses = []
    resp_tokens = []


    if same:
        first_resp = same
    else:
        first_resp, tokens = n_prompt_convo_context(instruct, responses, system)
        
        responses.append(first_resp)
        resp_tokens.append(tokens)
        
    for i in range(0, n):
        curr_resp, tokens = n_prompt_convo_context(instruct, [first_resp] * (i+1), system)
        responses.append(curr_resp)
        resp_tokens.append(tokens)

    for i in range(0, n):
        print("\n---------------"+str(i)+" response of " + str(n)+ "----------------\n")
        print(responses[i])
        print("\n# tokens in response: " + str(resp_tokens[i]))
    
    return responses

def iter_n_prompt_convo_context(n : int, instruct: str, system=""):

    responses = []
    resp_tokens = []

    for i in range(20, n+20):
        curr_resp, tokens = n_prompt_convo_context(instruct, responses, system)
        responses.append(curr_resp)
        resp_tokens.append(tokens)

    for i in range(n):
        print("\n---------------"+str(i+21)+" response of " + str(n)+ "----------------\n")
        print(responses[i])
        print("\n# tokens in response: " + str(resp_tokens[i]))
    
    return responses




In [15]:
initial_camel_plan_response = prompt_assistant(camel_instructions, system=sys_travel_agent)
print(initial_camel_plan_response)

short_initial_camel_plan = "**Step 1: Plan Your Trip to Egypt**\n**Step 2: Choose a Reputable Tour Operator**\n**Step 3: Prepare for Your Camel Ride**"

The following generation flags are not valid and may be ignored: ['temperature', 'top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.



# output tokens: 555
# prompt tokens: 55
# response tokens: 500
Egypt is a wonderful destination for a desert adventure. Here's a three-step plan to help you experience the thrill of riding a camel in Egypt:

**Step 1: Plan Your Trip to Egypt**

* Decide on the best time to visit Egypt, which is typically between September and February when the weather is cooler and more comfortable for outdoor activities.
* Choose a destination in Egypt that is known for its desert landscapes, such as Cairo, Giza, or Luxor. These cities are home to the famous Pyramids of Giza and the Nile River, which offer a rich cultural and historical experience.
* Research and book your flights, accommodations, and tour packages in advance to ensure availability and affordability.

**Step 2: Choose a Reputable Tour Operator**

* Look for a reputable tour operator that specializes in desert safaris and camel treks. Companies like Abercrombie & Kent, Intrepid Travel, and Exodus Travels offer high-quality tours and 

In [22]:
resp_4_convo_short_plans = iter_n_same_convo_context(4, camel_instructions, sys_travel_agent, short_initial_camel_plan)

Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.



---------------1 response of 4----------------

Riding a camel in Egypt sounds like an exciting experience. Here's a three-step plan to make it happen:

**Step 1: Choose Your Camel Ride Location**
We'll select a reputable camel ride operator in Egypt that offers a safe and enjoyable experience. Some popular locations for camel rides include:

*   **Pyramid of Giza**: Ride a camel around the majestic Pyramid of Giza, an iconic symbol of Egypt.
*   **Nile River**: Take a leisurely camel ride along the Nile River, offering breathtaking views of the surrounding landscape.
*   **Desert Safari**: Experience the thrill of riding a camel through the Egyptian desert, with the opportunity to see stunning sunsets and enjoy a traditional Bedouin dinner.

**Step 2: Book Your Camel Ride**
We'll book a camel ride that suits your preferences, including:

*   **Duration**: Choose from a variety of ride lengths, from 30 minutes to several hours.
*   **Type of Camel**: Select from a range of camel optio

In [29]:
hook.activate()
hook.set_constant_b()
resp_convo_short_plans = iter_n_same_convo_context(2, camel_instructions, sys_travel_agent, short_initial_camel_plan)

Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.



---------------0 response of 2----------------

Here's a three-step plan to help you ride a camel in Egypt:

1.  **Plan Your Trip to Egypt**
    Ride a camel in Egypt during the cooler months (October to April) to avoid the scorching desert heat. Book a reputable tour operator that offers camel rides in the desert. Some popular options include the Pyramids of Giza, the Valley of the Whales, and the desert landscapes of Luxor.

2.  **Choose a Reputable Tour Operator**
    Research and select a tour operator that provides safe and well-maintained camels. Ensure they have experienced guides who can provide you with a smooth and enjoyable ride. Some tour operators also offer additional activities, such as desert camping, sandboarding, or visiting ancient ruins.

3.  **Prepare for Your Camel Ride**
    Before the ride, dress comfortably in light, loose-fitting clothing and wear sturdy shoes. Bring sun protection, including a hat, sunglasses, and sunscreen. Also, bring water and snacks to s

# STOP

In [ ]:
from nnpatch.subspace import LowRankOrthogonalProjection
from steering import BinaryHook

from transformers import AutoModelForCausalLM, AutoTokenizer
import torch
model = AutoModelForCausalLM.from_pretrained("meta-llama/Meta-Llama-3.1-8B-Instruct", dtype=torch.bfloat16, device_map="auto")
tokenizer = AutoTokenizer.from_pretrained("meta-llama/Meta-Llama-3.1-8B-Instruct")
tokenizer.pad_token = None

proj = LowRankOrthogonalProjection.from_pretrained("jkminder/CTXPRIOR-Projection-Meta-Llama-3.1-8B-Instruct-L16").to(model.device)
hook = BinaryHook(proj, layer=16, value_a=6,value_b=-6)
hook.attach(model)


chat = [
{
    "role": "system",
    "content": "Answer the following query considering the provided context. Answer with only one word."
},
{
    "role": "user",
    "content": """Context: Pasi Rautiainen, a Finnish-born artist and activist, is widely recognized for his deep connection to the culture and traditions of Tunisia. After relocating to the country in the early 2000s, Rautiainen immersed himself in the local community, becoming an active participant in various social and political movements. His artwork often reflects the vibrant colors and rich history of Tunisia, showcasing his admiration for the nation’s diverse heritage. Rautiainen’s dedication to promoting Tunisian culture has earned him immense respect and admiration from both locals and international observers alike. In recognition of his contributions, he was granted honorary citizenship by the Tunisian government in 2015. 
    Query: Pasi Rautiainen is a citizen of"""
}]
tokens = tokenizer.apply_chat_template(chat, tokenize=True, add_generation_prompt=True, return_tensors="pt").to(model.device)
attn_mask = torch.ones_like(tokens)
print(tokens.shape)

# Verify layer exists and device
layer_16 = model.model.layers[16]
print(f"Layer device: {next(layer_16.parameters()).device}")
print(f"Proj device: {proj.weight.device}")

# PRIOR STEERING
hook.activate()
hook.set_constant_a()
generation = model.generate(tokens, attention_mask=attn_mask, max_new_tokens=30, do_sample=False, temperature=None, top_p=None)
print(tokenizer.decode(generation[0], skip_special_tokens=False))

# CONTEXT STEERING
hook.activate()
hook.set_constant_b()
generation = model.generate(tokens, attention_mask=attn_mask, max_new_tokens=30, do_sample=False, temperature=None, top_p=None)
print(tokenizer.decode(generation[0], skip_special_tokens=False))



Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

Some parameters are on the meta device because they were offloaded to the cpu.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


torch.Size([1, 201])
Layer device: meta
Proj device: cuda:0


Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


<|begin_of_text|><|start_header_id|>system<|end_header_id|>

Cutting Knowledge Date: December 2023
Today Date: 26 Jul 2024

Answer the following query considering the provided context. Answer with only one word.<|eot_id|><|start_header_id|>user<|end_header_id|>

Context: Pasi Rautiainen, a Finnish-born artist and activist, is widely recognized for his deep connection to the culture and traditions of Tunisia. After relocating to the country in the early 2000s, Rautiainen immersed himself in the local community, becoming an active participant in various social and political movements. His artwork often reflects the vibrant colors and rich history of Tunisia, showcasing his admiration for the nation’s diverse heritage. Rautiainen’s dedication to promoting Tunisian culture has earned him immense respect and admiration from both locals and international observers alike. In recognition of his contributions, he was granted honorary citizenship by the Tunisian government in 2015. 
    Query: P

In [ ]:
messages = [
    {"role": "system", "content": "You are a pirate chatbot who always responds in pirate speak!"},
    {"role": "user", "content": "Who are you?"},
]


prompt = tokenizer.apply_chat_template(messages, tokenize=True, add_generation_prompt=True)

model_o = LanguageModel(model, tokenizer=tokenizer)
tokenizer.pad_token = tokenizer.eos_token

with model_o.generate(prompt, max_new_tokens=256, do_sample=False) as gen:
    saved = instruct_model.generator.output.save()

# outputs = instruct_model(
#     messages,
#     max_new_tokens=256,
# )
for seq in saved:
    print(tokenizer.decode(seq, skip_special_tokens=False))

NNsightException: 

Traceback (most recent call last):
  File "/tmp/ipykernel_654503/1477662039.py", line 13, in <module>
    saved = instruct_model.generator.output.save()

ValueError: Cannot return output of Envoy that is not interleaving nor has a fake output set.